In [7]:
import json
from pathlib import Path
from typing import List
from tqdm import tqdm

import requests

In [8]:
SA_QUESTION_FILES = {
    "DoS":   Path("DoS_sa_qa/questions/dos_sa_questions.json"),
    "Fuzzy": Path("Fuzzy_sa_qa/questions/fuzzy_sa_questions.json"),
    "Gear":  Path("Gear_sa_qa/questions/gear_sa_questions.json"),
    "RPM":   Path("RPM_sa_qa/questions/rpm_sa_questions.json"),
}

SELECTED_DATASETS = ["RPM"]



In [9]:
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
MODEL_ID = "qwen2.5:14b"
MODEL_TAG = MODEL_ID.replace(":", "_").replace("-", "_").replace(".", "_")

print(f"Ollama URL: {OLLAMA_BASE_URL}")
print(f"Model: {MODEL_ID}")



Ollama URL: http://127.0.0.1:11434
Model: qwen2.5:14b


In [10]:
import re

SA_OUTPUT_TYPE = {
    "attack_ratio": "percentage",
    "attack_pattern": "short_text",
    "dominant_id": "can_id",
    "id_concentration": "short_text",
    "fps": "number",
    "traffic_shift": "short_text",
    "max_gap": "number",
    "timing_cause": "short_text",
    "dominant_var": "number",
    "payload_behavior": "short_text",
    "dlc_mode": "integer",
    "dlc_behavior": "short_text",
    "missing_count": "integer",
    "critical_behavior": "short_text",
    "out_of_range_id_count": "integer",
    "plausibility_relation": "short_text",
    "duplicate_count": "integer",
    "timestamp_cause": "short_text",
    "attack_type": "short_text",
    "evidence_pattern": "short_text",
}

SA_TYPE_HINT = {
    "percentage": "Answer with a percentage only, e.g., 12.5%.",
    "can_id": "Answer with one CAN ID only, uppercase hex format, e.g., 0x1AF.",
    "integer": "Answer with one non-negative integer only.",
    "number": "Answer with one numeric value only.",
    "short_text": "Answer with a short phrase only (1-4 words), not a full sentence.",
}


def build_sa_prompt_text(context: str, question: str, output_type: str = "short_text", strict_mode: bool = False) -> str:
    """Short-answer prompt with type-only constraints (no fixed labels)."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis assistant. "
        "Answer only with the final result. "
        "Do not output reasoning, analysis, steps, or markdown."
    )

    type_hint = SA_TYPE_HINT.get(output_type, SA_TYPE_HINT["short_text"])
    strict_tail = (
        "\nType enforcement: strictly follow the answer type hint only."
        if strict_mode else ""
    )

    return (
        f"{system_prompt}\n\n"
        "Analyze the following CAN bus time-window log and answer the question.\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Answer type hint:\n{type_hint}{strict_tail}\n\n"
        "Final answer (one line only):"
    )


def normalize_text(text: str) -> str:
    return " ".join(text.strip().split()).lower() if text else ""


def _sanitize_answer(text: str) -> str:
    lines = [ln.strip().strip("\"'`") for ln in (text or "").splitlines() if ln.strip()]
    if not lines:
        return ""
    return lines[-1]


def _extract_candidate_by_type(text: str, output_type: str) -> str:
    text = text or ""
    lines = [ln.strip().strip("\"'`") for ln in text.splitlines() if ln.strip()]

    if output_type == "can_id":
        m = re.search(r"0x[0-9A-F]+", text)
        return m.group(0) if m else ""

    if output_type == "integer":
        for ln in reversed(lines):
            m = re.search(r"(?<![\d.])-?\d+(?![\d.])", ln)
            if m and not m.group(0).startswith("-"):
                return m.group(0)
        return ""

    if output_type == "percentage":
        m = re.search(r"-?\d+(?:\.\d+)?%", text)
        return m.group(0) if m else ""

    if output_type == "number":
        for ln in reversed(lines):
            m = re.search(r"-?\d+(?:\.\d+)?", ln)
            if m:
                return m.group(0)
        return ""

    if not lines:
        return ""
    return " ".join(lines[-1].split()[:4])


def _is_valid_answer_format(ans: str, output_type: str) -> bool:
    if not ans:
        return False

    if output_type == "can_id":
        return re.fullmatch(r"0x[0-9A-F]+", ans) is not None

    if output_type == "integer":
        return re.fullmatch(r"\d+", ans) is not None

    if output_type == "percentage":
        if not ans.endswith("%"):
            return False
        try:
            float(ans[:-1])
            return True
        except Exception:
            return False

    if output_type == "number":
        try:
            float(ans)
            return True
        except Exception:
            return False

    words = ans.split()
    return 1 <= len(words) <= 4


def _extract_ollama_text(response_json: dict) -> str:
    if isinstance(response_json, dict):
        if isinstance(response_json.get("response"), str):
            return response_json["response"]
        if "message" in response_json and isinstance(response_json["message"], dict):
            content = response_json["message"].get("content")
            if isinstance(content, str):
                return content
    return ""


def query_sa_llm(context: str, question: str, sa_type: str = "", max_new_tokens: int = 192) -> str:
    output_type = SA_OUTPUT_TYPE.get(sa_type, "short_text")

    def _run_once(strict_mode: bool):
        prompt_text = build_sa_prompt_text(
            context, question, output_type=output_type, strict_mode=strict_mode
        )
        payload = {
            "model": MODEL_ID,
            "prompt": prompt_text,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": max_new_tokens,
            },
        }
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        completion = _extract_ollama_text(resp.json())
        return _sanitize_answer(completion), completion

    ans, raw1 = _run_once(strict_mode=False)
    if _is_valid_answer_format(ans, output_type):
        return ans

    cand1 = _extract_candidate_by_type(raw1, output_type)
    if _is_valid_answer_format(cand1, output_type):
        return cand1

    retry_ans, raw2 = _run_once(strict_mode=True)
    if _is_valid_answer_format(retry_ans, output_type):
        return retry_ans

    cand2 = _extract_candidate_by_type(raw2, output_type)
    if _is_valid_answer_format(cand2, output_type):
        return cand2

    return ""


def load_questions(path: Path) -> List[dict]:
    """Load questions from .json (list) or .jsonl."""
    if not path.exists():
        return []
    if path.suffix == ".json":
        with path.open("r", encoding="utf-8") as f:
            return json.load(f)
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records



In [11]:
# Function to generate explanation for model's own answer

def _sanitize_explanation(text: str, max_words: int = 10) -> str:
    text = " ".join((text or "").strip().split())
    if not text:
        return ""
    return " ".join(text.split()[:max_words])


def build_explanation_prompt(context: str, question: str, model_answer: str) -> str:
    """Generate prompt for explaining why the model's answer is correct."""
    system_prompt = (
        "You are a CAN-bus anomaly analysis expert. "
        "Provide one short explanation in max 10 words. "
        "No reasoning steps."
    )

    return (
        f"{system_prompt}\n\n"
        f"CAN log:\n{context}\n\n"
        f"Question:\n{question}\n\n"
        f"Answer:\n{model_answer}\n\n"
        "Explanation (max 10 words, one line only):"
    )


def generate_explanation_for_answer(context: str, question: str, model_answer: str, max_tokens: int = 24) -> str:
    """Generate explanation for why model's answer is correct."""
    if not model_answer:
        return ""

    prompt = build_explanation_prompt(context, question, model_answer)

    payload = {
        "model": MODEL_ID,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.0,
            "num_predict": max_tokens,
        },
    }

    try:
        resp = requests.post(
            f"{OLLAMA_BASE_URL}/api/generate",
            json=payload,
            timeout=180,
        )
        resp.raise_for_status()
        explanation = _extract_ollama_text(resp.json())
        return _sanitize_explanation(explanation, max_words=10)
    except Exception as e:
        print(f"[ERROR] Failed to generate explanation: {e}")
        return ""



In [12]:
RESUME_EXISTING = True  # If True, skip qa_id values already present in output JSONL.

for ds_name in SELECTED_DATASETS:
    q_path = SA_QUESTION_FILES[ds_name]
    if not q_path.exists():
        print(f"[WARN] Short-answer questions for {ds_name} not found at {q_path}, skip.")
        continue

    questions = load_questions(q_path)
    print(f"[INFO] {ds_name}: loaded {len(questions)} short-answer questions.")

    out_dir = q_path.parent.parent / "llm_answers"
    out_dir.mkdir(parents=True, exist_ok=True)
    ans_path = out_dir / f"{ds_name.lower()}_sa_answers_{MODEL_TAG}.jsonl"

    existing_ids = set()
    if RESUME_EXISTING and ans_path.exists():
        with ans_path.open("r", encoding="utf-8") as rf:
            for line in rf:
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except Exception:
                    continue
                qid = rec.get("qa_id")
                if qid is not None:
                    existing_ids.add(qid)

    pending_questions = [q for q in questions if q.get("qa_id") not in existing_ids]
    print(
        f"[INFO] {ds_name}: existing={len(existing_ids)} pending={len(pending_questions)} "
        f"(resume={RESUME_EXISTING and ans_path.exists()})"
    )

    if not pending_questions:
        print(f"[INFO] {ds_name}: nothing to do, skipping.")
        continue

    mode = "a" if (RESUME_EXISTING and ans_path.exists()) else "w"
    with ans_path.open(mode, encoding="utf-8") as f:
        for rec in tqdm(pending_questions, desc=f"{ds_name} SA answering+explaining"):
            qa_id = rec.get("qa_id")
            metadata = rec.get("metadata", {})
            dataset = metadata.get("dataset", ds_name)
            context = rec["context"]
            question = rec["question"]
            gt = rec.get("ground_truth", rec.get("answer", ""))

            pred = query_sa_llm(context, question, sa_type=rec.get("sa_type", ""))
            explanation = generate_explanation_for_answer(context, question, pred)

            answer_rec = {
                "qa_id": qa_id,
                "dataset": dataset,
                "sa_type": rec.get("sa_type"),
                "model": MODEL_ID,
                "llm_answer": pred,
                "llm_explanation": explanation,
                "ground_truth": gt,
                "is_exact_match": normalize_text(pred) == normalize_text(gt) if gt else False,
            }
            f.write(json.dumps(answer_rec, ensure_ascii=False) + "\n")

    print(f"[INFO] {ds_name}: SA answers + explanations saved to {ans_path}")


[INFO] RPM: loaded 1155 short-answer questions.
[INFO] RPM: existing=755 pending=400 (resume=True)


RPM SA answering+explaining: 100%|██████████| 400/400 [11:39:33<00:00, 104.93s/it]   

[INFO] RPM: SA answers + explanations saved to RPM_sa_qa\llm_answers\rpm_sa_answers_qwen2_5_14b.jsonl
